In [112]:
import pandas as pd

df = pd.read_csv('data/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [113]:
target_col = 'diagnosis'

In [114]:
def clean_data(df):
    print(f"Initial shape: {df.shape}")
    
    # Duplicates
    dup_count = df.duplicated().sum()
    if dup_count > 0:
        df = df.drop_duplicates()
        print(f"Removed {dup_count} duplicate rows.")
    
    # Drop fully-empty columns
    empty_cols = df.columns[df.isnull().all()]
    if len(empty_cols) > 0:
        df = df.drop(columns=empty_cols)
        print(f"Dropped fully-empty columns: {list(empty_cols)}")
    
    # Drop id-like columns (common naming patterns)
    id_cols = [col for col in df.columns if col.lower() in ['id', 'patient_id', 'index']]
    if id_cols:
        df = df.drop(columns=id_cols)
        print(f"Dropped ID columns: {id_cols}")
    
    # Report remaining missing values
    null_counts = df.isnull().sum()
    cols_with_nulls = null_counts[null_counts > 0]
    if len(cols_with_nulls) > 0:
        print(f"Columns with remaining missing values:\n{cols_with_nulls}")
    else:
        print("No missing values remaining.")
    
    print(f"Final shape: {df.shape}")
    return df

In [115]:
df=clean_data(df)

Initial shape: (569, 33)
Dropped fully-empty columns: ['Unnamed: 32']
Dropped ID columns: ['id']
No missing values remaining.
Final shape: (569, 31)


In [116]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 31 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   diagnosis                569 non-null    str    
 1   radius_mean              569 non-null    float64
 2   texture_mean             569 non-null    float64
 3   perimeter_mean           569 non-null    float64
 4   area_mean                569 non-null    float64
 5   smoothness_mean          569 non-null    float64
 6   compactness_mean         569 non-null    float64
 7   concavity_mean           569 non-null    float64
 8   concave points_mean      569 non-null    float64
 9   symmetry_mean            569 non-null    float64
 10  fractal_dimension_mean   569 non-null    float64
 11  radius_se                569 non-null    float64
 12  texture_se               569 non-null    float64
 13  perimeter_se             569 non-null    float64
 14  area_se                  569 non-null

In [117]:
# Cell: define target column (reusable variable, not hardcoded everywhere)
target_col = 'diagnosis'

# Cell: define the function (if not already defined)
def encode_categoricals(df, target_col):
    feature_cols = [col for col in df.columns if col != target_col]
    categorical_feature_cols = df[feature_cols].select_dtypes(include='object').columns.tolist()
    
    if categorical_feature_cols:
        print(f"One-hot encoding: {categorical_feature_cols}")
        df = pd.get_dummies(df, columns=categorical_feature_cols)
    else:
        print("No categorical features to encode.")
    
    return df

# Cell: call it
df = encode_categoricals(df, target_col)

No categorical features to encode.


In [118]:
def check_imbalance(df, target_col):
    counts = df[target_col].value_counts()
    percentages = df[target_col].value_counts(normalize=True) * 100
    
    print("Class counts:")
    print(counts)
    print("\nClass percentages:")
    print(percentages.round(2))
    
    minority_pct = percentages.min()
    if minority_pct < 10:
        print(f"\n⚠️ Severe imbalance detected (minority class: {minority_pct:.1f}%). Consider resampling or class_weight='balanced'.")
    elif minority_pct < 30:
        print(f"\n⚠️ Mild-to-moderate imbalance (minority class: {minority_pct:.1f}%). Use class_weight='balanced' and check precision/recall, not just accuracy.")
    else:
        print(f"\n Classes are reasonably balanced (minority class: {minority_pct:.1f}%).")
    
    return counts

check_imbalance(df, target_col)

Class counts:
diagnosis
B    357
M    212
Name: count, dtype: int64

Class percentages:
diagnosis
B    62.74
M    37.26
Name: proportion, dtype: float64

 Classes are reasonably balanced (minority class: 37.3%).


diagnosis
B    357
M    212
Name: count, dtype: int64

In [119]:
df[target_col] = df[target_col].map({'M': 1, 'B': 0})
df[target_col].value_counts()

diagnosis
0    357
1    212
Name: count, dtype: int64

In [120]:
def check_outliers(df, target_col):
    feature_cols = [col for col in df.columns if col != target_col]
    outlier_summary = {}
    
    for col in feature_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
        if len(outliers) > 0:
            outlier_summary[col] = len(outliers)
    
    print(f"Columns with outliers (IQR method):")
    for col, count in sorted(outlier_summary.items(), key=lambda x: -x[1]):
        print(f"  {col}: {count} outliers")
    
    return outlier_summary

outlier_summary = check_outliers(df, target_col)

Columns with outliers (IQR method):
  area_se: 65 outliers
  radius_se: 38 outliers
  perimeter_se: 38 outliers
  area_worst: 35 outliers
  smoothness_se: 30 outliers
  compactness_se: 28 outliers
  fractal_dimension_se: 28 outliers
  symmetry_se: 27 outliers
  area_mean: 25 outliers
  fractal_dimension_worst: 24 outliers
  symmetry_worst: 23 outliers
  concavity_se: 22 outliers
  texture_se: 20 outliers
  concave points_se: 19 outliers
  concavity_mean: 18 outliers
  radius_worst: 17 outliers
  compactness_mean: 16 outliers
  compactness_worst: 16 outliers
  symmetry_mean: 15 outliers
  fractal_dimension_mean: 15 outliers
  perimeter_worst: 15 outliers
  radius_mean: 14 outliers
  perimeter_mean: 13 outliers
  concavity_worst: 12 outliers
  concave points_mean: 10 outliers
  texture_mean: 7 outliers
  smoothness_worst: 7 outliers
  smoothness_mean: 6 outliers
  texture_worst: 5 outliers


In [121]:
#splitting into features
X = df.drop(columns=[target_col])
y = df[target_col]

print(X.shape)
print(y.shape)

(569, 30)
(569,)


In [122]:
from sklearn.model_selection import train_test_split
def split_data(X, y, is_classification=True, test_size=0.2):
    if is_classification:
        return train_test_split(X, y, test_size=test_size, random_state=42, stratify=y)
    else:
        return train_test_split(X, y, test_size=test_size, random_state=42)

In [123]:
X_train, X_test, y_train, y_test = split_data(X, y)

In [124]:
print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")

Train shape: (455, 30)
Test shape: (114, 30)


In [125]:
def choose_scaler(outlier_summary, total_features):
    outlier_ratio = len(outlier_summary) / total_features
    if outlier_ratio > 0.3:
        print(f"Significant outliers detected ({len(outlier_summary)}/{total_features} features) → using RobustScaler")
        from sklearn.preprocessing import RobustScaler
        return RobustScaler()
    else:
        print("Few outliers detected → using StandardScaler")
        from sklearn.preprocessing import StandardScaler
        return StandardScaler()

scaler = choose_scaler(outlier_summary, total_features=X.shape[1])

Significant outliers detected (29/30 features) → using RobustScaler


In [126]:
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaling complete")
print(X_train_scaled.shape)
print(X_test_scaled.shape)

Scaling complete
(455, 30)
(114, 30)


In [127]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(class_weight='balanced', random_state=42)
model.fit(X_train_scaled, y_train)

print("Model trained")

Model trained


In [128]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

y_pred = model.predict(X_test_scaled)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall: {recall_score(y_test, y_pred):.4f}")
print(f"F1 Score: {f1_score(y_test, y_pred):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nFull Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.9737
Precision: 0.9756
Recall: 0.9524
F1 Score: 0.9639

Confusion Matrix:
[[71  1]
 [ 2 40]]

Full Report:
              precision    recall  f1-score   support

           0       0.97      0.99      0.98        72
           1       0.98      0.95      0.96        42

    accuracy                           0.97       114
   macro avg       0.97      0.97      0.97       114
weighted avg       0.97      0.97      0.97       114



In [129]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(class_weight='balanced', random_state=42)
rf_model.fit(X_train_scaled, y_train)

y_pred_rf = rf_model.predict(X_test_scaled)
print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_rf):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_rf):.4f}")
print(f"F1: {f1_score(y_test, y_pred_rf):.4f}")
print(confusion_matrix(y_test, y_pred_rf))

Accuracy: 0.9737
Precision: 1.0000
Recall: 0.9286
F1: 0.9630
[[72  0]
 [ 3 39]]


In [130]:
from sklearn.ensemble import GradientBoostingClassifier

gb_model = GradientBoostingClassifier(random_state=42)
gb_model.fit(X_train_scaled, y_train)

y_pred_gb = gb_model.predict(X_test_scaled)
print(f"Accuracy: {accuracy_score(y_test, y_pred_gb):.4f}")

print(f"Precision: {precision_score(y_test, y_pred_gb):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_gb):.4f}")
print(f"F1: {f1_score(y_test, y_pred_gb):.4f}")
print(confusion_matrix(y_test, y_pred_gb))


Accuracy: 0.9649
Precision: 1.0000
Recall: 0.9048
F1: 0.9500
[[72  0]
 [ 4 38]]
